# Data Preprocessing ML

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

#---------- Reproducibility Configuration
sys.path.insert(0, str(Path("src").resolve()))
from reproducibility import DEFAULT_SEED, configure_reproducibility

SEED = DEFAULT_SEED
configure_reproducibility(SEED, include_tensorflow=False)
print(f"Global reproducibility seed configured: {SEED}")


# ---------------- Paths
BASE_DIR = Path(".")
CLEAN_DIR = BASE_DIR / "Data/2-Consolidated-Time-Clean"
INPUT_CSV = CLEAN_DIR / "emg_consolidated_time_clean.csv"

# >>> Output inside Data
ML_DIR = BASE_DIR / "Data/3-Data-ML-1"
ML_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Utilities
def median_dt_seconds(t: pd.Series) -> float:
    t = pd.to_numeric(t, errors="coerce").dropna().values
    if len(t) < 2:
        return np.nan
    dt = np.diff(t)
    dt = dt[~np.isnan(dt)]
    return float(np.nanmedian(dt)) if len(dt) else np.nan

def excel_max_rows_per_sheet(include_header: bool = True) -> int:
    MAX = 1_048_576
    return MAX - (1 if include_header else 0)

def save_all_formats(df: pd.DataFrame, stem: str):
    csv_path = ML_DIR / f"{stem}.csv"
    txt_path  = ML_DIR / f"{stem}.txt"
    xlsx_path = ML_DIR / f"{stem}.xlsx"

    df.to_csv(csv_path, index=False)
    df.to_csv(txt_path,  index=False, sep="\t")

    max_rows_excel = excel_max_rows_per_sheet(include_header=True)
    with pd.ExcelWriter(xlsx_path) as writer:
        if len(df) <= max_rows_excel:
            df.to_excel(writer, index=False, sheet_name="All")
        else:
            start = 0; part = 1
            while start < len(df):
                end = min(start + max_rows_excel, len(df))
                df.iloc[start:end].to_excel(writer, index=False, sheet_name=f"Part_{part}")
                start = end; part += 1

    print(f"Saved -> {csv_path}")
    print(f"Saved -> {txt_path}")
    print(f"Saved -> {xlsx_path}")

def make_feature_names(samples_per_win: int):
    cols = []
    for a in S_aliases:  # S1..S7
        for i in range(1, samples_per_win + 1):
            cols.append(f"{a}_{i:05d}")
    return cols

# ---------------- Load
df = pd.read_csv(INPUT_CSV, dtype={"user": str, "gesture": str})
required_cols = {"user", "gesture", "Time", "sensor", "value"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns in {INPUT_CSV}: {sorted(missing)}")

# ---------------- S1..S7 aliases (stable and global)
# Alphabetical order for reproducible consistency.
sensors = sorted(df["sensor"].dropna().unique().tolist())
if len(sensors) != 7:
    raise ValueError(f"Expected 7 sensors and found {len(sensors)}: {sensors}")

S_aliases = [f"S{i}" for i in range(1, 8)]
SENSOR_ALIAS = {sensors[i]: S_aliases[i] for i in range(7)}  # real name -> S#
ALIAS_TO_SENSOR = {v: k for k, v in SENSOR_ALIAS.items()}    # S# -> real name

# Save the mapping for traceability
pd.DataFrame(
    [{"alias": a, "sensor_name": ALIAS_TO_SENSOR[a]} for a in S_aliases]
).to_csv(ML_DIR / "sensors_alias_map.csv", index=False)
print("Alias map (S# -> sensor):")
for a in S_aliases:
    print(f"  {a} -> {ALIAS_TO_SENSOR[a]}")

# ---------------- Datasets to build (non-overlapping windows)
WINDOWS_S = [2.00, 1.00, 0.50, 0.25, 0.15, 0.05, 0.02, 0.01]   # seconds

# ---------------- Validation/accumulators
build_logs = []   # per window and per (user, gesture) -> build_summary.csv
strict_logs = []  # strict time/chronology checks -> time_checks.csv

def _check_time_integrity(wide: pd.DataFrame, user: str, gesture: str):
    """Record time/order validations for one (user, gesture)."""
    times = pd.to_numeric(wide["Time"], errors="coerce").dropna()
    is_monotonic = bool(times.is_monotonic_increasing)
    has_dupes = bool(times.duplicated().any())
    dt_med = median_dt_seconds(times)
    duration_s = float(times.iloc[-1] - times.iloc[0]) if len(times) else np.nan
    # NA values per sensor (should be 0 after dropna)
    na_counts = {SENSOR_ALIAS[s]: int(wide[s].isna().sum()) for s in sensors}
    strict_logs.append({
        "user": user, "gesture": gesture,
        "n_rows": len(times), "is_monotonic": is_monotonic, "has_dupes_Time": has_dupes,
        "dt_median": dt_med, "t_start": float(times.iloc[0]) if len(times) else np.nan,
        "t_end": float(times.iloc[-1]) if len(times) else np.nan,
        "duration_s": duration_s, **{f"na_{SENSOR_ALIAS[s]}": na_counts[SENSOR_ALIAS[s]] for s in sensors}
    })

def build_dataset_for_window(win_s: float) -> pd.DataFrame:
    """
    Builds one dataset for a win_s window in seconds.
    - Works independently by (user, gesture).
    - Requires the 7 sensors and aligns them on common timestamps (pivot inner join + dropna).
    - Creates non-overlapping windows and concatenates the 7 sensors -> S#_00001..S#_NNNNN.
      Chronology guarantee: 'wide' is sorted by Time ASC; slices are contiguous
      and sensor-wise flattening preserves temporal order (oldest to newest).
    """
    rows_meta, rows_data = [], []

    feature_cols = None
    samples_per_win_ref = None

    # Iterate independently by user and gesture
    for (user, gesture), grp in df.groupby(["user", "gesture"], sort=False):
        # Wide format: one column per sensor
        wide = (grp.pivot_table(index="Time", columns="sensor", values="value", aggfunc="first")
                  .reset_index()
                  .sort_values("Time")
                  .reset_index(drop=True))

        # Check that all 7 sensors are present
        if not set(sensors).issubset(set(wide.columns)):
            missing_s = [s for s in sensors if s not in wide.columns]
            build_logs.append({
                "window_s": win_s, "user": user, "gesture": gesture,
                "status": "SKIPPED", "reason": f"missing_sensors:{missing_s}"
            })
            continue

        # Keep only timestamps with data in all 7 sensors (strict alignment)
        wide = wide[["Time"] + sensors].copy()
        # Chronological order guaranteed
        wide = wide.sort_values("Time").reset_index(drop=True)
        # Collapse exact duplicate timestamps
        if wide["Time"].duplicated(keep="first").any():
            wide = wide[~wide["Time"].duplicated(keep="first")].reset_index(drop=True)
        # Align: keep ONLY timestamps where all 7 sensors exist
        before = len(wide)
        wide = wide.dropna(subset=sensors).reset_index(drop=True)
        after = len(wide)
        dropped_not_common = before - after

        # Temporal integrity check/log
        _check_time_integrity(wide, user, gesture)

        # Validation metrics by group
        dt = median_dt_seconds(wide["Time"])
        if not np.isfinite(dt) or dt <= 0:
            build_logs.append({
                "window_s": win_s, "user": user, "gesture": gesture,
                "status": "SKIPPED", "reason": "invalid_dt", "rows_after": len(wide)
            })
            continue

        # Estimated sampling rate (prefer a stable integer with tolerance)
        hz_est = 1.0 / dt
        hz_round = int(round(hz_est))
        if hz_round > 0 and abs(hz_est - hz_round) / hz_round <= 0.01:  # ±1%
            hz = hz_round
        else:
            hz = max(1, int(round(hz_est)))

        samples_per_win = int(round(win_s * hz))
        if samples_per_win <= 1:
            build_logs.append({
                "window_s": win_s, "user": user, "gesture": gesture,
                "status": "SKIPPED", "reason": f"window_too_small_{samples_per_win}_samples",
                "hz_est": hz_est, "hz_used": hz, "dt": dt, "rows_after": len(wide)
            })
            continue

        # Feature names (the first group defines the global reference)
        if feature_cols is None:
            feature_cols = make_feature_names(samples_per_win)
            samples_per_win_ref = samples_per_win

        # If other sampling rates produce another 'samples_per_win', adjust with padding/trimming
        group_cols = feature_cols if samples_per_win == samples_per_win_ref else make_feature_names(samples_per_win)

        # Create non-overlapping windows by group (user/gesture)
        total_samples = len(wide)
        nwin = total_samples // samples_per_win
        remainder = total_samples % samples_per_win
        if nwin == 0:
            build_logs.append({
                "window_s": win_s, "user": user, "gesture": gesture,
                "status": "NO_WINDOWS", "hz_used": hz, "dt": dt,
                "rows_after": len(wide), "samples_per_win": samples_per_win,
                "dropped_not_common": dropped_not_common
            })
            continue

        # Extract sensor matrix and flatten by sensor (S1..S7), preserving temporal order
        X = wide[sensors].to_numpy(dtype=float)  # [N, 7]
        t0 = float(wide["Time"].iloc[0]); t1 = float(wide["Time"].iloc[-1])

        for w in range(nwin):
            sl = slice(w * samples_per_win, (w + 1) * samples_per_win)
            seg = X[sl, :]  # [samples_per_win, 7] in chronological order
            # Sensor-wise flattening (all S1 points, then S2, ..., S7)
            vec = np.concatenate([seg[:, i] for i in range(7)], axis=0)

            # Pad/trim if it differs from the global schema
            if len(group_cols) != len(vec):
                if len(vec) > len(group_cols):
                    vec = vec[:len(group_cols)]
                else:
                    pad = np.full(len(group_cols) - len(vec), np.nan, dtype=float)
                    vec = np.concatenate([vec, pad], axis=0)

            rows_meta.append((user, gesture))
            rows_data.append(vec)

        # Validation log by (user, gesture)
        build_logs.append({
            "window_s": win_s, "user": user, "gesture": gesture, "status": "OK",
            "hz_used": hz, "dt": dt,
            "rows_after": len(wide), "samples_per_win": samples_per_win,
            "n_windows": nwin, "remainder": remainder,
            "t_start": t0, "t_end": t1, "duration_s": t1 - t0,
            "dropped_not_common": dropped_not_common
        })

    # Build final DataFrame
    if not rows_data:
        return pd.DataFrame(columns=["user", "gesture"])

    X_all = np.vstack(rows_data)
    meta_df = pd.DataFrame(rows_meta, columns=["user", "gesture"])
    feat_df = pd.DataFrame(X_all, columns=feature_cols if feature_cols else group_cols)

    # Metadata first and sorted by user/gesture for presentation only
    out = pd.concat([meta_df, feat_df], axis=1).sort_values(["user", "gesture"]).reset_index(drop=True)
    return out

# ---------------- Build and save the 6 packages
for win_s in WINDOWS_S:
    ds = build_dataset_for_window(win_s)
    print(f"\nWindow {win_s:.2f}s -> shape: {ds.shape}  (rows, cols)")
    stem = f"ml_win_{win_s:.2f}s"
    save_all_formats(ds, stem)

# ---------------- Save validation reports
summary_df = pd.DataFrame(build_logs).sort_values(["window_s", "user", "gesture"]).reset_index(drop=True)
summary_df.to_csv(ML_DIR / "build_summary.csv", index=False)

time_checks_df = pd.DataFrame(strict_logs).sort_values(["user", "gesture"]).reset_index(drop=True)
time_checks_df.to_csv(ML_DIR / "time_checks.csv", index=False)

print("\nValidation saved to:")
print(" -", (ML_DIR / "build_summary.csv").resolve())
print(" -", (ML_DIR / "time_checks.csv").resolve())
print("\nDone. Files in:", ML_DIR.resolve())
